In [ ]:
# NB3 - Cell 1: Mount Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/CLASS_AlignED_MVP"

In [ ]:
# NB3 - Cell 3: Define workspace paths
import os, json, shutil
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive/CLASS_AlignED_MVP"
MANIFEST = f"{PROJECT_ROOT}/processed/manifest.json"
WORKSPACE = f"{PROJECT_ROOT}/processed/graphrag_workspace"

Path(WORKSPACE).mkdir(parents=True, exist_ok=True)

with open(MANIFEST, "r", encoding="utf-8") as r:
    manifest = json.load(r)

print("Workspace:", WORKSPACE)

In [ ]:
# NB3 - Cell 4: Initialize GraphRAG workspace (only run once)
%cd {WORKSPACE}
!graphrag init

In [ ]:
# NB3 - Cell 5: Copy syllabi extracted text into GraphRAG input/
import json

INPUT_DIR = f"{WORKSPACE}/input"
Path(INPUT_DIR).mkdir(parents=True, exist_ok=True)

for doc in manifest:
    # read extracted text JSON from NB1 output
    text_json_path = doc["text_json"]
    with open(text_json_path, "r", encoding="utf-8") as r:
        text_obj = json.load(r)
    out_txt = f"{INPUT_DIR}/{doc['doc_id']}_{Path(doc['source_file']).stem}.txt"
    with open(out_txt, "w", encoding="utf-8") as w:
        w.write(text_obj["text"])
    print("Wrote input:", out_txt)

In [ ]:
# NB3 - Cell 6: Run indexing
%cd {WORKSPACE}
!graphrag index

In [ ]:
# NB3 - Cell 7: Run a couple of test queries
%cd {WORKSPACE}
!graphrag query "Summarize the grading structure and major assessments for these syllabi."
!graphrag query "What are the learning outcomes and how are they assessed?" --method local

In [ ]:
# NB3 - Cell 8 (optional): Gradio wrapper around graphrag query
!pip -q install gradio

import subprocess, textwrap, gradio as gr

def ask(q, method="local"):
    cmd = ["graphrag", "query", q, "--method", method]
    p = subprocess.run(cmd, cwd=WORKSPACE, capture_output=True, text=True)
    if p.returncode != 0:
        return f"ERROR:\n{p.stderr}"
    return p.stdout

demo = gr.Interface(
    fn=ask,
    inputs=[gr.Textbox(label="Question"), gr.Dropdown(["local","global","drift"], value="local", label="Method")],
    outputs=gr.Textbox(label="Answer"),
    title="CLASS AlignED MVP (GraphRAG Query)"
)
demo.launch(share=True)